<a href="https://colab.research.google.com/github/Yurida26/UTS-NLP-SemanticSquad/blob/main/3_Feature_Engineering_and_Machine_Learning_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Notebook ini mencakup:
1. **Feature Extraction:** Implementasi TF-IDF dan Word Embedding (Word2Vec).
2. **Modeling:** Eksperimen menggunakan Naive Bayes, Logistic Regression, dan SVM.
3. **Comparison:** Perbandingan performa antar metode fitur dan algoritma.

Load Dataset & Library

In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 19.9 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Load data hasil preprocessing
try:
    df = pd.read_csv('products_cleaned.csv')
    df = df.dropna()
    print(f"Dataset berhasil dimuat: {len(df)} baris")
except:
    print("Error: File 'products_cleaned.csv' tidak ditemukan. Pastikan sudah upload file hasil Tahap 1.")

df.head()

Dataset berhasil dimuat: 12000 baris


,product_name,review_text,product_category,text_combined,clean_text,category_label
0,"MT.EDMA CYCLONE 54"" Kipas Angin Dekorasi BK Ba...",Brang datang dengan selamat. masih nunggu jadw...,Elektronik,"MT.EDMA CYCLONE 54"" Kipas Angin Dekorasi BK Ba...",mtedma cyclone kipas angin dekorasi bk baling ...,0
1,Kabel Ties / Cable Ties Stainless Steel SS 4.6...,kualitas baik akurasi baik delivery tepat waktu,Elektronik,Kabel Ties / Cable Ties Stainless Steel SS 4.6...,kabel ties cable ties stainless steel ss x mm ...,0
2,HANDY TALKY WINLAN WLN W-218 (KD-C170) UHF ISI...,"barang sesuai deskripsi, berfungsi baik, dan a...",Elektronik,HANDY TALKY WINLAN WLN W-218 (KD-C170) UHF ISI...,handy talky winlan wln w kdc uhf isi pcs hitam...,0
3,BRACKET LOOKTECH 32-55 inch Untuk Led TV,"Cepat,murah,dekat rumah juga",Elektronik,BRACKET LOOKTECH 32-55 inch Untuk Led TV Cepat...,bracket looktech inch led tv cepatmurahdekat r...,0
4,LG FV1285S3VS - MESIN CUCI FRONT LOADING LG IN...,setelah 2 hari pemakaian. test durability 2 ha...,Elektronik,LG FV1285S3VS - MESIN CUCI FRONT LOADING LG IN...,lg fvsvs mesin cuci front loading lg inverter ...,0


**FEATURE ENGINEERING**
1. TF-IDF Vectorization

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Inisialisasi TF-IDF dengan N-Gram (1, 2)
# Menggunakan max_features=5000 untuk efisiensi namun tetap representatif
tfidf_vect = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)

# Transformasi teks menjadi fitur numerik
X_tfidf = tfidf_vect.fit_transform(df['clean_text'])

# Simpan objek vectorizer untuk tahap Deployment nanti
pickle.dump(tfidf_vect, open('tfidf_vectorizer.pkl', 'wb'))

print("Ekstraksi Fitur TF-IDF Selesai.")
print("Dimensi Matrix TF-IDF:", X_tfidf.shape)

Ekstraksi Fitur TF-IDF Selesai.
Dimensi Matrix TF-IDF: (12000, 5000)


2. Word Embedding (Word2Vec)

In [4]:
from gensim.models import Word2Vec

# Tokenisasi: Memecah kalimat menjadi list kata untuk input Word2Vec
sentences = [str(text).split() for text in df['clean_text']]

# Melatih model Word2Vec (vector_size=100)
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)

# Fungsi Vector Averaging untuk mendapatkan identitas satu ulasan
def get_avg_vector(text):
    words = str(text).split()
    vectors = [w2v_model.wv[w] for w in words if w in w2v_model.wv]
    if len(vectors) > 0:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(100)

# Mengubah seluruh dataset menjadi representasi Word Embedding
X_w2v = np.array([get_avg_vector(t) for t in df['clean_text']])

print("Ekstraksi Fitur Word2Vec (Embedding) Selesai.")
print("Dimensi Matrix Word2Vec:", X_w2v.shape)

Ekstraksi Fitur Word2Vec (Embedding) Selesai.
Dimensi Matrix Word2Vec: (12000, 100)


**MODELING**

1. Data Splitting & Pipeline Setup

In [5]:
# Label target (Kategori Produk)
y = df['category_label']

# Split data untuk fitur TF-IDF (80% Train, 20% Test)
X_train_tfidf, X_test_tfidf, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

# Split data untuk fitur Word2Vec
X_train_w2v, X_test_w2v, _, _ = train_test_split(X_w2v, y, test_size=0.2, random_state=42)

print("Split data berhasil dilakukan.")

Split data berhasil dilakukan.


2. Training & Evaluasi Model (6 Kombinasi)

In [6]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Kamus untuk menyimpan skor akurasi agar mudah dibandingkan
results = {}

print("--- EKSPERIMEN MODEL DENGAN TF-IDF ---")

# 1. Naive Bayes (TF-IDF)
nb_tfidf = MultinomialNB().fit(X_train_tfidf, y_train)
results['NB + TF-IDF'] = nb_tfidf.score(X_test_tfidf, y_test)

# 2. Logistic Regression (TF-IDF)
lr_tfidf = LogisticRegression(max_iter=1000).fit(X_train_tfidf, y_train)
results['LogReg + TF-IDF'] = lr_tfidf.score(X_test_tfidf, y_test)

# 3. SVM (TF-IDF)
svm_tfidf = SVC(kernel='linear').fit(X_train_tfidf, y_train)
results['SVM + TF-IDF'] = svm_tfidf.score(X_test_tfidf, y_test)

print("\n--- EKSPERIMEN MODEL DENGAN WORD2VEC ---")

# 4. Logistic Regression (Word2Vec)
lr_w2v = LogisticRegression(max_iter=1000).fit(X_train_w2v, y_train)
results['LogReg + Word2Vec'] = lr_w2v.score(X_test_w2v, y_test)

# 5. SVM (Word2Vec)
svm_w2v = SVC(kernel='linear').fit(X_train_w2v, y_train)
results['SVM + Word2Vec'] = svm_w2v.score(X_test_w2v, y_test)

# Menampilkan semua hasil
for model, score in results.items():
    print(f"{model}: {score:.4f}")

# Mencari model terbaik
best_model_name = max(results, key=results.get)
print(f"\nModel Terbaik: {best_model_name}")

--- EKSPERIMEN MODEL DENGAN TF-IDF ---

--- EKSPERIMEN MODEL DENGAN WORD2VEC ---
NB + TF-IDF: 0.9421
LogReg + TF-IDF: 0.9604
SVM + TF-IDF: 0.9688
LogReg + Word2Vec: 0.7442
SVM + Word2Vec: 0.7621

Model Terbaik: SVM + TF-IDF


**MENYIMPAN MODEL TERBAIK**


In [7]:
import pickle

# 1. Logika Otomatis untuk memilih objek model berdasarkan akurasi tertinggi
if best_model_name == 'NB + TF-IDF':
    final_model = nb_tfidf
elif best_model_name == 'LogReg + TF-IDF':
    final_model = lr_tfidf
elif best_model_name == 'SVM + TF-IDF':
    final_model = svm_tfidf
elif best_model_name == 'LogReg + Word2Vec':
    final_model = lr_w2v
else:
    final_model = svm_w2v

# 2. Simpan model terbaik ke file .pkl
pickle.dump(final_model, open('best_model_classifier.pkl', 'wb'))

print(f"SUKSES! File 'best_model_classifier.pkl' adalah hasil dari {best_model_name}")
print("Silakan download file 'best_model_classifier.pkl' dan 'tfidf_vectorizer.pkl' di sidebar kiri.")

SUKSES! File 'best_model_classifier.pkl' adalah hasil dari SVM + TF-IDF
Silakan download file 'best_model_classifier.pkl' dan 'tfidf_vectorizer.pkl' di sidebar kiri.
